# Ансамблирование моделей

При изучении Data Science идея ансамблирования впервые встречается при упоминании такой модели, как случайный лес. В данной модели обучаются базовые модели, представленные решающими деревьями,  предсказания которых впоследствии агрегируются некоторым образом, зависящим от задачи. В случае задачи регрессии берётся среднее либо средневзвешенное. В случае задачи классификации класс присваивается по принципу большинства.

### Существует три основных способа построения ансамблей:

* Бэггинг — параллельно обучаем множество одинаковых моделей, а для предсказания берём среднее по предсказаниям каждой из моделей.
* Бустинг — последовательно обучаем множество одинаковых моделей, где каждая новая модель концентрируется на тех примерах, где предыдущая допустила ошибку.
* Стекинг — параллельно обучаем множество разных моделей, отправляем их результаты в финальную модель (мета модель), и уже она принимает решение.

**Бэггинг (bagging)** — алгоритм построения ансамбля путём параллельного обучения множества независимых друг от друга моделей.

Самым распространённым примером **ансамбля** типа бэггинг является случайный лес (**Random Forest**). 

# Блендинг

Простейшая реализация стекинга заключается в блендинге (blending). 

Схематично блендинг можно представить следующим образом:

<img src='Images/ml_06.png'>

Суть **блендинга** состоит в следующем: предположим у нас есть обучающая выборка X, которую мы делим пополам. Первая часть используется для обучения базовых моделей, а на второй базовые модели делают предсказания – метапризнаки, на которых уже и обучается в дальнейшем **метамодель**.

Недостатки блендинга видны невооруженным глазом: ни базовые модели, ни метамодель не обучаются на полных данных.

# Стекинг

Для решения этой проблемы используется усовершенствованная модель блендинга, которая имеет полноценное название — стекинг. Идея борьбы с недостатком блендинга — использование **кросс-валидации**.

Рассмотрим как обучается классический стекинг. Пусть у нас есть таблица с примерами X и ответами на них y. Количество признаков — m, количество наблюдений — n, количество моделей в стекинге — K.

1. Обучающая выборка разбивается на L равных частей, называемых **фолдами**. Например, для трёх фолдов (L = 3) схематично это будет выглядеть следующим образом:

<img src='Images/ml_07.png'>

2. Затем для каждой базовой модели эти фолды перебираются следующим образом: на каждом шаге фиксируются L - 1 фолдов для обучения базовых моделей и один фолд для предсказания (в случае бинарной классификации каждая модель предсказывает вероятность принадлежности к классу 1, в случае мультиклассовой классификации — к каждому классу). В результате будет сформировано L предсказаний, из которых формируется метапризнак $M_j$, где j — номер модели:

<img src='Images/ml_08.png'>

Такой подход к формированию метапризнаков позволяет избежать переобучения. Действительно, можно рассматривать L - 1- фолд как обучающую выборку, а оставшийся — как тестовую. Таким образом, мы обучаемся на тренировочной выборке, но предсказания делаем для той выборки, которую ещё не видели.

3. После того как мы проделаем шаг 2 для всех базовых моделей, мы получим новый набор данных, состоящий из  K метапризнаков — предсказаний каждой из моделей. Предсказания моделей будут использоваться в качестве метапризнаков, на которых будет обучена метамодель.

Пусть мы взяли три разных модели, т.е. K = 3. Это будет выглядеть следующим образом:

<img src='Images/ml_09.png'>

Примечание. Кроме метафакторов, метамодель может использовать для своего обучения изначальные признаки из исходного набора данных. Иными словами, обычно **метамодель** обучается только на метапризнаках (на выходах моделей первого уровня), но можно дать ей дополнительно исходные признаки из оригинального набора данных (X), например возраст, доход, и т.д.

### Что это даёт:

* Метамодель получает дополнительную информацию.

* Может компенсировать потери, если метамодели первого уровня что-то "пропустили".

* Иногда улучшает качество ансамбля.

### Есть некоторые рекомендации, как правильно строить стекинг:

* В качестве метамоделей лучше всего применять простые модели: например, для задачи регрессии — линейную регрессию, а для задачи классификации — логистическую регрессию.

* В качестве базовых моделей лучшего всего использовать модели различной природы.

# Бустинг

**Бустинг (boosting)** — это алгоритм построения ансамбля, основанный на последовательном построении слабых моделей, причём каждая новая модель пытается уменьшить ошибку предыдущей. После того как все модели обучены, они объединяются в композицию.

Примечание: Под **слабыми моделями** мы подразумеваем модели, точность которых немногим выше, чем случайное угадывание. Как правило, это короткие деревья решений, они обладают слабой предсказательной способностью.

В отличие от бэггинга (параллельное выполнение моделей), бустинг обучается на одном и том же наборе данных, без генерации дополнительных выборок. Однако в процессе обучения меняются так называемые **веса наблюдений**. Если слабая модель допустила ошибку на каких-то примерах, то значимость (вес) этих примеров увеличивается и на них концентрируется следующая за ней модель.

Представить алгоритм бустинга можно следующей схемой:

<img src='Images/ml_10.png'>

Поскольку основная цель бустинга — уменьшение **смещения**, в качестве базовых моделей часто выбирают алгоритмы с высоким смещением и небольшим разбросом, например **короткие деревья решений**. У каждого из таких деревьев слабая предсказательная способность, но если их объединить, мы получим очень мощную модель.

# Адаптивный бустинг (Ada Boost)

Первая реализация бустинга называлась AdaBoost. Это модель, которая подразумевает воплощение той самой идеи взвешивания объектов, которую мы рассмотрели выше. Алгоритм предполагает постоянную модификацию объектов выборки путём их взвешивания, причём веса обновляются специальным образом: каждая новая модель из ансамбля обучается на взвешенных данных и обращает большее внимание на ошибки своих предшественников.

### Плюсы:

* Он прост. Операции просты в реализации и не требуют вычисления производных, умножений матриц и прочих сложных математических конструкций.

* Накладные расходы бустинга минимальны. Время построения определяется временем построения базовых моделей.

* Показывает хорошую обобщающую способность.

* Имеет возможность идентификации шумовых объектов в ряде случаев.

### Минусы:

* Жадное добавление алгоритмов приводит к неоптимальности композиции.

* Склонен к переобучению при наличии шума в данных.

* Алгоритм является эвристикой, и «взвешивание» объектов, на котором он основан, не подкреплено математическим обоснованием.

# Градиентный бустинг (Gradient Boosting)

Градиентный бустинг (Gradient Boosting, GB) — это наиболее обобщённая версия бустинга, закреплённая математическим обоснованием. Впервые алгоритм был опубликован профессором статистики Стэнфордского университета Джеромом Фридманом. Алгоритм оказался очень эффективным и в дальнейшем был множество раз модифицирован — до Extreme Gradient Boosting (XgBoost) и других модификаций, таких как CatBoost от Яндекса и LightGMB от Microsoft.

В GB принцип классического бустинга сохраняется: каждый последующий алгоритм улучшает предыдущий, но, в отличие эвристического «взвешивания» наблюдений, градиентный бустинг использует информацию о функции потерь для построения нового алгоритма.

В качестве базовой модели можно использовать всё что угодно, но общепринятым является использование деревьев решений. Практика показывает, что это наилучший выбор, так как деревья решений очень просты в построении и из всех слабых моделей обладают наилучшей способностью описывать сложные зависимости.

Бустинг, использующий в качестве базовой модели дерево решений, называется **градиентным бустингом над деревьями решений** (Gradient Boosting on Decision Trees, GBDT).

<img src='Images/ml_11.png'>

---

Основным преимуществом такой схемы градиентного бустинга является эффективность в поиске нелинейных зависимостей в сравнении с любыми моделями, основанными на решающих деревьях. Это преимущество стало причиной доминирования GBDT на огромном спектре соревнований — от кредитного скоринга до рекомендательных систем.

---

### Рекомендации по выбору внешних параметров алгоритма:

* Количество деревьев (n_estimators). Чем больше деревьев вы берёте, тем меньше ошибка на обучающем наборе данных, вплоть до 0, но, как вы понимаете, тем выше шанс переобучиться. Лучше начинать с небольшого количества моделей (50-100), а затем следить за ошибкой на тестовой выборке.
* Темп обучения  (learning_rate). Чем выше темп обучения, тем больше вклад каждого следующего дерева будет в модель и тем быстрее вы сойдётесь к минимуму функции потерь и сведёте ошибку к 0. Однако снова высок риск переобучения. Рекомендуемые значения — от 0.01 до 1.
* Максимальная глубина деревьев (max_depth). Градиентный бустинг лучше всего работает со слабыми моделями — это короткие деревья решений с глубиной от 1 до 8.

<img src='Images/ml_12.png'>

# Pipeline

Процесс автоматического поэтапного выполнения манипуляций с данными, включающий в себя сбор, обработку, генерацию и отбор признаков, обучение модели с последующей её настройкой и проверкой качества называется **пайплайном**.

**Основные цели использования пайплайнов** — автоматизация, ускорение вычислений с использованием многопоточности в Python и дальнейшее развертывание пайплайна для использования в периодических расчетах (сбор данных в режиме онлайн/онлайн-работа модели). Кроме того, пайплайны хороши в случае, когда надо подобрать оптимальные гиперпараметры для всего цикла обработки данных и последующего обучения.

# Metric Learning

Любые подходы в машинном обучении, которые требуют измерения расстояния между объектами в выборке, являются подходами metric learning (часто их обозначают как метрические алгоритмы). Основными задачами, решаемыми подходами metric learning, наряду с классическими задачами обучения с учителем, являются задача кластеризации и задача понижения размерности. Также metric learning иногда используется в задачах восстановления данных по принципу нахождения ближайшего похоже объекта.